# headswap_V2 - test: pre-edit donor expression before T4

**Run order:** Cell 1 (setup, restarts the kernel) -> Cell 2 (upload) -> Cell 3 (run).

Cell 1 detects whether this runtime already has ComfyUI installed:
- **Fresh runtime** -> full setup (clone, deps, ComfyUI, Krea2 weights).
- **Existing runtime** -> skips the slow install, just re-syncs the repo to the latest commit on this branch.

Cell 3 tests the new `pre_edit_donor_expression` step (docs/PIPELINE_STATE.md CHECKPOINT-11/12/13): it measures the **target's** actual expression, edits the **donor** photo to match it (pure Krea2 generation, no mask), then runs T4's existing two-step pass (main pass + face_refine) on the edited donor. The cell displays all three stages: the original donor face, the donor after the expression edit, and T4's final result.


In [ ]:
#@title Cell 1 - Setup (detects an existing runtime; only a fresh one gets the full install)
from pathlib import Path
import subprocess, shutil, os, signal, sys
import importlib.metadata as _im

assert Path("/content").exists(), "Open this notebook in Google Colab."

def _import_torch():
    import torch
    torch.cuda.is_available()  # touch a real attribute to force full init
    return torch

try:
    torch = _import_torch()
except AttributeError as exc:
    # Known Colab base-image hiccup, seen on a brand-new runtime's very
    # first `import torch`: the module partially initializes and a later
    # submodule (torch.fx via torch._export.verifier) is missing, raised
    # as "partially initialized module ... most likely due to a circular
    # import". Reinstalling the SAME pinned version (not upgrading, which
    # could pull a build mismatched with Colab's GPU driver) refreshes
    # whatever got corrupted.
    try:
        _torch_ver = _im.version("torch")
    except _im.PackageNotFoundError:
        _torch_ver = None
    pkg = f"torch=={_torch_ver}" if _torch_ver else "torch"
    print(f"torch import broken on this runtime ({exc}); reinstalling {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-deps", "--no-cache-dir", pkg],
                   check=True)
    for _m in list(sys.modules):
        if _m == "torch" or _m.startswith("torch."):
            del sys.modules[_m]
    torch = _import_torch()

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then re-run this cell.")
print(f"GPU {torch.cuda.get_device_name(0)}")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
BRANCH = "simple-full-body-head-swap"

# Fresh runtime: no ComfyUI on disk yet -> needs the full install below.
# Existing runtime (reconnect / re-run): ComfyUI + weights are already on
# disk -> only re-sync the repo, skip the slow scripts/setup_colab.sh.
FRESH_RUNTIME = not Path("/content/ComfyUI/server.py").exists()
if FRESH_RUNTIME:
    print("-> Fresh runtime detected (no ComfyUI on disk) - running full setup.")
else:
    print("-> Existing runtime detected (ComfyUI already on disk) - skipping "
          "the slow install, just re-syncing the repo.")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"),
      subprocess.getoutput(f"git -C {REPO} log -1 --pretty=%s"))

os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

if FRESH_RUNTIME:
    shutil.rmtree("/content/ComfyUI", ignore_errors=True)
    r = subprocess.run(["bash", "scripts/setup_colab.sh", "--no-drive", "--krea2"],
                       check=False, capture_output=True, text=True)
    print("setup exit:", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("--- stderr ---"); print(r.stderr[-2000:])
        raise SystemExit("setup_colab.sh failed")
else:
    print("ComfyUI + weights already present - skipping scripts/setup_colab.sh")

subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                "--no-deps", "numpy==2.4.6"], check=True)

print("\n✓ Setup complete. Restarting kernel (this is expected)...")
print("   When it comes back, run Cell 2.")
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
#@title Cell 2 - Upload YOUR two images
# Upload the BODY first (the photo you want to keep: pose, clothes, background;
# this is also where the DESIRED expression is measured from), then the FACE
# (the donor whose identity you want transferred in -- its expression will be
# edited by Cell 3 to match the body's before the swap).
import os
from pathlib import Path
from PIL import Image
from IPython.display import display
from google.colab import files

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "my_pair"
PAIR.mkdir(parents=True, exist_ok=True)
for old in PAIR.glob("*"):
    old.unlink()

def _grab(role):
    print(f"\n=== Upload the {role} image ===")
    up = files.upload()
    if not up:
        raise SystemExit(f"No {role} image uploaded - re-run this cell.")
    name = next(iter(up))
    dest = PAIR / f"{role}.png"
    Image.open(name).convert("RGB").save(dest)
    os.remove(name)
    im = Image.open(dest)
    print(f"saved {role}: {im.size[0]}x{im.size[1]}px")
    if max(im.size) < 500:
        print(f"   note: small source ({im.size[0]}x{im.size[1]}). The pipeline "
              "upscales to 1024 so generated detail survives, but a larger "
              "original will always look sharper.")
    return im

body_im = _grab("body")
face_im = _grab("face")

print("\n--- BODY (kept: pose / clothing / background / DESIRED expression) ---")
display(body_im)
print("--- FACE (donor: identity; expression will be edited to match BODY) ---")
display(face_im)
print("\n✓ Ready. Run Cell 3.")


In [ ]:
#@title Cell 3 - Run: identity_lora_strength sweep on T4's main pass
SEED = 46  #@param {type:"integer"}

# THE LEVER UNDER TEST (docs/PIPELINE_STATE.md CHECKPOINT-13 calls this "the
# only untested lever", and it is still untested -- the pre-edit detour never
# came back to it). The identity LoRA is trained to transplant the head from
# image 2, and a head includes its expression. This is the global dial on how
# hard it does that. T4's own value is 1.0.
#
# Sweep 1.0 -> 0.7 -> 0.5, one run each, same seed. Judge BOTH axes every
# time: did the expression move, and did the identity survive. The likely
# failure mode -- seen on every other lever in this investigation -- is that
# they move together, i.e. the smile relaxes only as the face stops being the
# donor. If that happens the global dial is the wrong shape of control and
# the answer is per-block or timestep-scheduled LoRA, not a different number.
IDENTITY_LORA_STRENGTH = 1.0  #@param {type:"number"}

# Donor pre-edit: CLOSED as a dead end (CHECKPOINT-14) -- four GPU rounds,
# zero expression movement, and round 4 was verified unconfounded. Left here
# only so the arm can be re-run for reference; leave it False.
RUN_DONOR_PRE_EDIT = False  #@param {type:"boolean"}
USE_CODE_DEFAULTS = True  #@param {type:"boolean"}
PRE_EDIT_DENOISE = 0.45  #@param {type:"number"}
PRE_EDIT_REF_BOOST = 2.0  #@param {type:"number"}
PRE_EDIT_CFG = 4.0  #@param {type:"number"}
PRE_EDIT_DISABLE_LORA = True  #@param {type:"boolean"}

import sys, os, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
os.chdir(REPO)

import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

PAIR = REPO / "data" / "custom" / "my_pair"
body_path, face_path = PAIR / "body.png", PAIR / "face.png"
if not (body_path.exists() and face_path.exists()):
    raise SystemExit("Images missing - run Cell 2 first.")

body_im = Image.open(body_path).convert("RGB")
face_im = Image.open(face_path).convert("RGB")

runtime = get_shared_krea2_runtime(init_custom_nodes=True)
cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
cfg.update({
    "seed": int(SEED),
    "save_debug": False,
    "verbose": False,
    # Pre-step under test (default OFF in the yaml): edit the DONOR's
    # expression to match the TARGET's measured expression before T4's own
    # main pass + face_refine run. See docs/PIPELINE_STATE.md CHECKPOINT-13.
    "pre_edit_donor_expression": bool(RUN_DONOR_PRE_EDIT),
    # The lever under test. Bundles are cached per (unet, clip, vae, lora,
    # STRENGTH), so each sweep value loads its own bundle -- no stale reuse.
    "identity_lora_strength": float(IDENTITY_LORA_STRENGTH),
})
print(f"identity_lora_strength={IDENTITY_LORA_STRENGTH}  (T4 default 1.0)")
if RUN_DONOR_PRE_EDIT and not USE_CODE_DEFAULTS:
    cfg.update({
        "pre_edit_donor_expression_denoise": float(PRE_EDIT_DENOISE),
        "pre_edit_donor_expression_ref_boost": float(PRE_EDIT_REF_BOOST),
        "pre_edit_donor_expression_cfg": float(PRE_EDIT_CFG),
        "pre_edit_donor_expression_disable_lora": bool(PRE_EDIT_DISABLE_LORA),
    })
    print("knobs: hand-tuned from the Cell 3 form (USE_CODE_DEFAULTS=False)")
else:
    print("knobs: using code defaults synced by Cell 1")

OUT_DIR = REPO / "results" / "pre_edit_expression_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
result = create_pipeline(cfg, runtime=runtime).run(body_im, face_im, out_dir=OUT_DIR)
elapsed = time.perf_counter() - t0

meta = result.meta or {}
pre_edit = meta.get("pre_edit_donor_expression") or {}
route = meta.get("body_route") or {}
edit_mode = meta.get("edit_mode")
route_name = route.get("route")
print(f"\n{elapsed:.0f}s  mode={edit_mode}  route={route_name}  out={result.image.size}")
print(f"pre_edit_donor_expression: applied={pre_edit.get('applied')} reason={pre_edit.get('reason')}")
if pre_edit.get("target_expression"):
    te = pre_edit["target_expression"]
    print(f"  target expression measured: {te.get('label')} "
          f"(smile_ratio={te.get('smile_ratio')} open_ratio={te.get('open_ratio')})")
    print(f"  donor edit knobs: denoise={pre_edit.get('denoise')} "
          f"ref_boost={pre_edit.get('ref_boost')} cfg={pre_edit.get('cfg')} "
          f"steps={pre_edit.get('steps')} "
          f"identity_lora_disabled={pre_edit.get('identity_lora_disabled')}")

pre_edit_face_path = OUT_DIR / "debug_pre_edit_donor_face.png"

display(Markdown("### 1 - Donor face (uploaded)"))
display(face_im)

if pre_edit.get("applied") and pre_edit_face_path.is_file():
    display(Markdown(
        "### 2 - Donor after the expression edit "
        "(pure Krea2 generation, no mask -- BEFORE T4's two-step pass)"
    ))
    display(Image.open(pre_edit_face_path))
else:
    skip_reason = pre_edit.get("reason")
    display(Markdown(
        f"### 2 - Donor expression edit SKIPPED ({skip_reason}) "
        "-- T4 ran on the original donor face"
    ))

display(Markdown(
    "### 3 - Final result (T4's two-step pass -- main pass + face_refine "
    "-- using the edited donor above)"
))
display(result.image)

final_path = OUT_DIR / "final_result.png"
result.image.save(final_path)
print(f"\nSaved: {final_path}")
if pre_edit_face_path.is_file():
    print(f"Saved: {pre_edit_face_path}")

# Uncomment to download:
# from google.colab import files; files.download(str(final_path))


In [ ]:
#@title Cell 4 - Expression-transfer PROBE (one Krea2 sample, no T4)
# Inverts the one mechanism this model does reliably. CHECKPOINTs 11-14 spent
# four sessions trying to STOP image 2's expression from riding along with its
# identity, across seven levers, and never once succeeded. So stop fighting it
# and point it the other way:
#
#     image 1 (scene)  = the DONOR   -> the identity we want to KEEP
#     image 2 (person) = the TARGET  -> the expression we want to TAKE
#
# If it works, the output is the donor wearing the target's expression, which
# is exactly the donor image T4 wants as input.
#
# The roles are swapped IN CODE. Upload normally in Cell 2 (body = target,
# face = donor) -- do not swap the uploads by hand.
#
# Identity LoRA defaults OFF: transplanting a whole head from image 2 is what
# it is trained to do, and here that is the failure mode, not the goal.
# THE EXPRESSION YOU WANT, in plain words, stated as a FACT about the
# picture being made. Round 1 of this probe asked the model to "match the
# expression of the person in the second image" -- a meta-instruction about
# which input to obey, which CHECKPOINT-11 already measured as inert twice
# and which was inert here too (identity held perfectly, expression did not
# move). Stating the expression as fact is the only phrasing shape with a
# working precedent in this repo, and it also sidesteps the broken openness
# measurement (CHECKPOINT-14) -- you know the expression you want.
#
# Leave blank to fall back to the old image-2 phrasing (not recommended).
PROBE_EXPRESSION = "not smiling, with a closed mouth and a neutral, serious expression"  #@param {type:"string"}

# Round 1 held identity but moved nothing. These are the "make the transfer
# actually happen" values: more image-2 influence, more room to move, and
# the LoRA back on -- it is possible the image2 -> image1 transfer we could
# never suppress IS the LoRA, in which case turning it off removed the very
# mechanism this probe is trying to exploit.
PROBE_LORA_STRENGTH = 0.5  #@param {type:"number"}
PROBE_DENOISE = 0.6  #@param {type:"number"}
PROBE_REF_BOOST = 3.5  #@param {type:"number"}
PROBE_CFG = 4.0  #@param {type:"number"}
SEED = 46  #@param {type:"integer"}

import sys, os
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
os.chdir(REPO)

import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

PAIR = REPO / "data" / "custom" / "my_pair"
body_path, face_path = PAIR / "body.png", PAIR / "face.png"
if not (body_path.exists() and face_path.exists()):
    raise SystemExit("Images missing - run Cell 2 first.")

target_im = Image.open(body_path).convert("RGB")   # expression source
donor_im = Image.open(face_path).convert("RGB")    # identity to keep

runtime = get_shared_krea2_runtime(init_custom_nodes=True)
cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
cfg.update({
    "seed": int(SEED),
    "verbose": False,
    "probe_expression_text": str(PROBE_EXPRESSION).strip(),
    "probe_expression_lora_strength": float(PROBE_LORA_STRENGTH),
    "probe_expression_denoise": float(PROBE_DENOISE),
    "probe_expression_ref_boost": float(PROBE_REF_BOOST),
    "probe_expression_cfg": float(PROBE_CFG),
})

OUT_DIR = REPO / "results" / "expr_probe"
pipe = create_pipeline(cfg, runtime=runtime)
res = pipe.probe_expression_transfer(
    identity_img=donor_im,       # image 1 -> keep this face
    expression_img=target_im,    # image 2 -> take this expression
    out_dir=OUT_DIR,
)

print(f"\n{res['latency_s']}s  lora_strength={res['lora_strength']} "
      f"denoise={res['denoise']} ref_boost={res['ref_boost']} cfg={res['cfg']}")
print(f"prompt: {res['prompt']}")

display(Markdown("### 1 - IDENTITY to keep (donor, = image 1 / scene)"))
display(donor_im)
display(Markdown("### 2 - EXPRESSION to take (target, = image 2 / person)"))
display(target_im)
display(Markdown("### 3 - PROBE OUTPUT"))
display(res["image"])

display(Markdown(
    "**Judge two things, they are separate questions:**\n\n"
    "1. Did the **expression** move toward image 2?\n"
    "2. Is it still **image 1's person**?\n\n"
    "Both yes -> this is the expression step; feed the output to T4 as the donor.\n\n"
    "Expression moved but it is now image 2's face -> the channel carries "
    "identity too; raising cfg / lowering ref_boost is the next thing to try.\n\n"
    "Nothing moved -> the transfer only works through the LoRA, and this "
    "route is closed like the others."
))


In [ ]:
#@title Cell 5 - LivePortrait: change ONLY the expression (installs on first run)
# Purpose-built expression transfer, driven by the target photo's own dense
# keypoints -- no text in the loop at all, which is the bottleneck that killed
# every Krea2 attempt (CHECKPOINT-11..14, eight levers, five sessions).
#
#   SOURCE  = the face whose identity we KEEP   (donor, or a T4 output)
#   DRIVING = the photo whose expression we TAKE (the target)
#
# ANIMATION_REGION is the reason this tool fits the requirement: "lip"
# animates the mouth ONLY and leaves the eye region untouched -- a parameter,
# not a prompt the model may ignore.
#
# LICENSE: LivePortrait code + weights are MIT. It uses InsightFace buffalo_l
# for face analysis, which is non-commercial research only -- but headswap_V2
# ALREADY depends on buffalo_l (see scripts/download_insightface.py and every
# run log), so this adds no new restriction. That pre-existing InsightFace
# constraint is worth raising separately for anything shipping commercially.

SOURCE_IS = "donor"  #@param ["donor", "t4_output"]

# "lip" animates the MOUTH ONLY and leaves the eye region alone. "exp" is the
# fuller expression but can move the eyes, so it does not meet the "don't
# touch the eyes" requirement.
ANIMATION_REGION = "lip"  #@param ["lip", "exp", "eyes", "pose", "all"]

# MUST be False when driving from a single IMAGE.
#
# Relative motion transfers the DELTA between the driving frame and the
# driving sequence's own reference frame. With a video that is what you want.
# With one image, that image is both the reference AND the target, so the
# delta is ~zero and almost nothing transfers -- measured here: a full 32s
# run that returned a near-copy of the source. Absolute motion makes the
# source adopt the driving image's actual expression instead.
RELATIVE_MOTION = False  #@param {type:"boolean"}

# Amplifies the transferred motion. Raise to 1.2-1.5 if the change is real
# but too subtle; drop below 1.0 if the mouth overshoots into a grimace.
DRIVING_MULTIPLIER = 1.0  #@param {type:"number"}
STITCHING = True  #@param {type:"boolean"}

import os, sys, subprocess, glob, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

LP = Path("/content/LivePortrait")
REPO = Path("/content/headswap_V2")

# ---- install (idempotent) -------------------------------------------------
if not (LP / "inference.py").exists():
    print("-> cloning LivePortrait ...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/KwaiVGI/LivePortrait", str(LP)], check=True)

    # NEVER `pip install -r requirements.txt` here: it pins torch/numpy and
    # would wreck the Krea2 environment in this same runtime. This repo has
    # already been bitten by exactly that (simple-lama silently downgraded
    # pillow/numpy and killed rembg + restore_background with no error).
    # Install only what LivePortrait needs that Colab does not already have.
    print("-> installing curated deps (NOT requirements.txt) ...")
    subprocess.run(["pip", "install", "-q", "--no-cache-dir",
                    "tyro", "imageio", "imageio-ffmpeg", "rich", "pykalman",
                    "ffmpeg-python"], check=False)
    # Re-pin numpy to the value Cell 1 set, in case a transitive dep moved it.
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                    "--no-deps", "numpy==2.4.6"], check=False)
else:
    print("LivePortrait already cloned - skipping install")

WEIGHTS = LP / "pretrained_weights"
if not (WEIGHTS / "liveportrait").exists():
    print("-> downloading weights (~500MB, humans only) ...")
    # Python API, not the CLI: `huggingface-cli` was removed in favour of
    # `hf`, and hard-coding either name breaks again on the next rename.
    # HF_HUB_DISABLE_XET mirrors what scripts/setup_colab.sh already does --
    # Xet transfers stall on Colab.
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    from huggingface_hub import snapshot_download
    _repo_err = None
    for _repo in ("KwaiVGI/LivePortrait", "KlingTeam/LivePortrait"):
        try:
            snapshot_download(
                repo_id=_repo,
                local_dir=str(WEIGHTS),
                ignore_patterns=["*animal*"],
            )
            print(f"   weights from {_repo}")
            _repo_err = None
            break
        except Exception as exc:  # noqa: BLE001
            _repo_err = exc
            print(f"   {_repo} failed ({type(exc).__name__}: {exc}); trying next")
    if _repo_err is not None:
        raise SystemExit(f"weights download failed: {_repo_err}")
else:
    print("weights already present - skipping download")

# ---- inputs ---------------------------------------------------------------
PAIR = REPO / "data" / "custom" / "my_pair"
driving_path = PAIR / "body.png"          # target = expression to TAKE
if SOURCE_IS == "donor":
    source_path = PAIR / "face.png"
else:
    source_path = REPO / "results" / "pre_edit_expression_test" / "final_result.png"
    if not source_path.exists():
        raise SystemExit("No T4 output yet - run Cell 3 first, or pick SOURCE_IS='donor'.")
for f in (source_path, driving_path):
    if not f.exists():
        raise SystemExit(f"missing {f} - run Cell 2 first.")

OUT = Path("/content/lp_out")
OUT.mkdir(parents=True, exist_ok=True)
for old in OUT.glob("*"):
    if old.is_file():
        old.unlink()

# ---- run ------------------------------------------------------------------
os.chdir(LP)
if str(LP) not in sys.path:
    sys.path.insert(0, str(LP))
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

from src.config.argument_config import ArgumentConfig
from src.config.inference_config import InferenceConfig
from src.config.crop_config import CropConfig
from src.live_portrait_pipeline import LivePortraitPipeline

def partial_fields(target_class, kwargs):
    return target_class(**{k: v for k, v in kwargs.items() if hasattr(target_class, k)})

args = ArgumentConfig(
    source=str(source_path),
    driving=str(driving_path),
    output_dir=str(OUT),
    flag_relative_motion=bool(RELATIVE_MOTION),
    animation_region=str(ANIMATION_REGION),
    flag_stitching=bool(STITCHING),
    flag_pasteback=True,
    driving_multiplier=float(DRIVING_MULTIPLIER),
)
pipe = LivePortraitPipeline(
    inference_cfg=partial_fields(InferenceConfig, args.__dict__),
    crop_cfg=partial_fields(CropConfig, args.__dict__),
)
t0 = time.perf_counter()
pipe.execute(args)
print(f"\n{time.perf_counter()-t0:.1f}s  region={ANIMATION_REGION} "
      f"relative={RELATIVE_MOTION} mult={DRIVING_MULTIPLIER}")

os.chdir(REPO)

# ---- show -----------------------------------------------------------------
produced = sorted(OUT.glob("**/*"), key=lambda f: f.stat().st_mtime if f.is_file() else 0)
produced = [f for f in produced if f.is_file()]
print("produced:", [f.name for f in produced])

display(Markdown("### 1 - SOURCE (identity kept)"))
display(Image.open(source_path))
display(Markdown("### 2 - DRIVING (expression taken)"))
display(Image.open(driving_path))

imgs = [f for f in produced if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
if imgs:
    display(Markdown(f"### 3 - LIVEPORTRAIT OUTPUT (`{imgs[-1].name}`)"))
    display(Image.open(imgs[-1]))
    final = REPO / "results" / "liveportrait_result.png"
    final.parent.mkdir(parents=True, exist_ok=True)
    Image.open(imgs[-1]).convert("RGB").save(final)
    print(f"saved -> {final}")
else:
    vids = [f for f in produced if f.suffix.lower() == ".mp4"]
    if vids:
        display(Markdown(f"### 3 - output came out as VIDEO (`{vids[-1].name}`) "
                         "- extracting the first frame"))
        import imageio.v3 as iio
        frame = iio.imread(vids[-1], index=0)
        im = Image.fromarray(frame)
        display(im)
        final = REPO / "results" / "liveportrait_result.png"
        final.parent.mkdir(parents=True, exist_ok=True)
        im.convert("RGB").save(final)
        print(f"saved -> {final}")
    else:
        print("No output found - check the log above.")

display(Markdown(
    "**Judge:** did the mouth/expression move toward image 2, while the "
    "identity AND the eye region stayed as image 1?\n\n"
    "- **nothing moved** -> RELATIVE_MOTION must be **False** for a single "
    "driving image (with one image the relative delta is ~zero)\n"
    "- expression moved but too subtle -> raise DRIVING_MULTIPLIER to 1.2-1.5\n"
    "- mouth overshoots / grimaces -> lower DRIVING_MULTIPLIER below 1.0\n"
    "- eyes moved too -> ANIMATION_REGION must be 'lip', not 'exp'/'all'\n"
    "- a visible seam around the face -> STITCHING is off, turn it back on"
))
